# Generate Paper Figures

This notebook reproduces **Figure 2** and **Figure 3** from
*Identifying and Measuring Satellite Streaks in DECam Images*.

Figures are saved as publication-quality PDFs in the `figures/` directory.

In [ ]:
import sys
import os

# Path setup
SATMETRICS_PATH = os.path.expanduser(
    "~/Documents/papers/RECA-2025_DECam_satellites/satmetrics"
)
sys.path.insert(0, SATMETRICS_PATH)

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, REPO_ROOT)

FIGURES_DIR = os.path.join(REPO_ROOT, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import requests
from astropy.io import fits
from astropy.utils.data import download_file
import cv2

import line_detection_updated as ld
import image_rotation as ir
import pixelplot

from reca_streaks.photometry import streak_photometry, gaussian_1d
from reca_streaks.data import retrieve_hdu_image

In [ ]:
# Publication plot defaults
plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

In [ ]:
def load_decam_image(expnum, detector):
    """Load a DECam image and return (image_data, header, header_expnum)."""
    natroot = "https://astroarchive.noirlab.edu"
    adsurl = f"{natroot}/api/adv_search"
    jj = {
        "outfields": ["md5sum", "archive_filename", "EXPNUM"],
        "search": [
            ["instrument", "decam"],
            ["proc_type", "instcal"],
            ["EXPNUM", expnum, expnum],
            ["prod_type", "image"],
        ],
    }
    apiurl = f"{adsurl}/find/?limit=1"
    data = requests.post(apiurl, json=jj).json()
    if len(data) < 2:
        raise ValueError(f"No image found for expnum {expnum}")
    md5sum = data[1]["md5sum"]
    access_url = f"{natroot}/api/retrieve/{md5sum}/?hdus={detector}"
    filename = download_file(access_url, cache=True)
    hdu_list = fits.open(filename)
    return hdu_list[1].data, hdu_list[1].header, hdu_list[0].header

In [ ]:
def detect_lines_hough(
    image,
    threshold=0.075,
    flux_prop_thresholds=[0.1, 0.2, 0.3, 1],
    blur_kernel_sizes=[3, 5, 9, 11],
    brightness_cuts=(2, 2),
    thresholding_cut=0.5,
    **kwargs,
):
    """Run Hough Transform line detection via satmetrics."""
    lineDetector = ld.LineDetection(image)
    lineDetector.threshold = threshold
    lineDetector.flux_prop_thresholds = flux_prop_thresholds
    lineDetector.blur_kernel_sizes = blur_kernel_sizes
    lineDetector.brightness_cuts = brightness_cuts
    lineDetector.thresholding_cut = thresholding_cut
    for key, value in kwargs.items():
        setattr(lineDetector, key, value)
    lineDetector.detect()
    return lineDetector.results

---
## Figure 2 — Line Detection Pipeline

Three-panel figure showing the thresholded image, Canny edges, and the
rotated streak cutout with median profile.

In [ ]:
# Load image for Figure 2  (adjust expnum/detector as needed)
expnum_fig2 = 1103448
detector_fig2 = 26

image_fig2, header_fig2, header_expnum_fig2 = load_decam_image(
    expnum_fig2, detector_fig2
)
detected_lines_fig2 = detect_lines_hough(image_fig2, threshold=0.05)

In [ ]:
# Rotate the streak horizontal
clustered_fig2 = ld.cluster(
    detected_lines_fig2["Cartesian Coordinates"],
    detected_lines_fig2["Lines"],
)
rotated_fig2, best_fit_fig2 = ir.complete_rotate_image(
    clustered_lines=clustered_fig2,
    angles=detected_lines_fig2["Angles"],
    image=image_fig2,
    cart_coord=detected_lines_fig2["Cartesian Coordinates"],
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Thresholded image
axes[0].imshow(detected_lines_fig2["Thresholded Image"], cmap="gray")
axes[0].set_title("Thresholded Image")
axes[0].set_xlabel("X (pixels)")
axes[0].set_ylabel("Y (pixels)")

# Panel 2: Canny edges
axes[1].imshow(detected_lines_fig2["Edges"], cmap="gray")
axes[1].set_title("Canny Edge Detection")
axes[1].set_xlabel("X (pixels)")
axes[1].set_ylabel("Y (pixels)")

# Panel 3: Rotated streak cutout
rotated_img = rotated_fig2[0]
axes[2].imshow(
    rotated_img,
    origin="lower",
    cmap="gray",
    vmin=np.percentile(rotated_img, 5),
    vmax=np.percentile(rotated_img, 95),
    aspect="auto",
)
axes[2].set_title("Rotated Streak Cutout")
axes[2].set_xlabel("X (pixels)")
axes[2].set_ylabel("Y (pixels)")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "figure2_detection_pipeline.pdf"))
plt.show()
print("Figure 2 saved.")

---
## Figure 3 — Streak Profile and Photometry (NAVSTAR-70)

Two-panel figure: (top) 1-D cross-streak profile with Gaussian fit and
on/off-streak boundaries; (bottom) 2-D image with coloured region mask.

In [ ]:
# NAVSTAR-70: exposure 1134933, CCD 5
expnum_fig3 = 1134933
detector_fig3 = 5

image_fig3, header_fig3, header_expnum_fig3 = load_decam_image(
    expnum_fig3, detector_fig3
)
detected_lines_fig3 = detect_lines_hough(image_fig3)

clustered_fig3 = ld.cluster(
    detected_lines_fig3["Cartesian Coordinates"],
    detected_lines_fig3["Lines"],
)
rotated_fig3, best_fit_fig3 = ir.complete_rotate_image(
    clustered_lines=clustered_fig3,
    angles=detected_lines_fig3["Angles"],
    image=image_fig3,
    cart_coord=detected_lines_fig3["Cartesian Coordinates"],
)

In [ ]:
# Run aperture photometry (returns dict with intermediates)
hdu_list_fig3 = retrieve_hdu_image(expnum_fig3, detector_fig3)

result = streak_photometry(
    rotated_fig3[0],
    hdu_list=hdu_list_fig3,
    make_plots=False,
)

print(f"SB = {result['sb_arcsec']:.2f} +/- {result['sb_arcsec_err']:.2f} counts/arcsec^2")
print(f"SNR = {result['snr']:.1f}")
if result["sb_mag"] is not None:
    print(f"SB = {result['sb_mag']:.3f} +/- {result['sb_mag_err']:.3f} mag/arcsec^2")

In [ ]:
# Build Figure 3
fig, axes = plt.subplots(2, 1, figsize=(10, 10), gridspec_kw={"height_ratios": [1, 1]})

# --- Top panel: 1-D profile ---
ax = axes[0]
y = result["y"]
profile = result["profile_y"]
fp = result["fit_params"]

ax.plot(y, profile, color="black", label="1D Profile")

# Gaussian model overlay
y_model = np.linspace(y.min(), y.max(), 500)
ax.plot(
    y_model,
    gaussian_1d(y_model, fp["A"], fp["y0"], fp["sigma"], fp["offset"]),
    color="orange",
    linewidth=2,
    label="Gaussian fit",
    zorder=5,
)

# On-streak boundaries (red dashed)
ax.axvline(result["on_ymin"], color="red", linestyle="--", linewidth=2,
           label="On-streak", zorder=10)
ax.axvline(result["on_ymax"], color="red", linestyle="--", linewidth=2,
           zorder=10)

# Off-streak boundaries (blue dashed)
ax.axvline(result["off1_ymin"], color="blue", linestyle="--", label="Off-streak")
ax.axvline(result["off1_ymax"], color="blue", linestyle="--")
ax.axvline(result["off2_ymin"], color="blue", linestyle="--")
ax.axvline(result["off2_ymax"], color="blue", linestyle="--")

ax.set_xlabel("Y (pixels)")
ax.set_ylabel("Mean Counts")
ax.set_title("Streak Profile and Regions (NAVSTAR-70)")
ax.legend()

# --- Bottom panel: 2-D image with region mask ---
ax2 = axes[1]
img = result["image_data"]
rm = result["region_mask"]

ax2.imshow(
    img,
    origin="lower",
    cmap="gray",
    vmin=np.percentile(img, 5),
    vmax=np.percentile(img, 99),
)
ax2.imshow(
    np.ma.masked_where(rm == 0, rm),
    origin="lower",
    cmap="coolwarm",
    alpha=0.5,
)
ax2.set_xlabel("X (pixels)")
ax2.set_ylabel("Y (pixels)")
ax2.set_title("On-streak (red) and Off-streak (blue) Regions")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "figure3_streak_profile.pdf"))
plt.show()
print("Figure 3 saved.")

---
Both figures have been saved to the `figures/` directory as PDFs.